# Example 12.4.1 / 2: Enumeration versus generation

Companion notebook for **12.4.1 Cutting Stock Example** in the AMPL lab Documentation, section "Enumeration versus generation". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Enumeration versus generation

Enumeration is attractive when all feasible patterns are few: it yields one ordinary master model and an easy completeness check. Generation avoids storing an exponential pattern set but needs a correct iterative loop and a pricing problem solved to optimality.

On the same instance, enumeration lists every pattern that fits a roll, solves the master LP and
then the integer master over all patterns. The run sets `relax_integrality` with
`ampl.option[...]` and reads the plan with `to_dict()`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS;                                          # item types
param width {ITEMS} > 0;                            # item width
param stock_width > 0;                              # width of one stock roll
param demand {ITEMS} integer >= 0;                  # pieces ordered per item
param nPatterns integer >= 0 default 0;             # number of enumerated patterns
set PATTERNS = 1..nPatterns;                        # every feasible pattern
param pieces {ITEMS, PATTERNS} integer >= 0;        # pieces of item i in pattern p

var UsePattern {PATTERNS} integer >= 0;             # rolls cut with pattern p

minimize Rolls: sum {p in PATTERNS} UsePattern[p];  # stock rolls used
subject to MeetDemand {i in ITEMS}:                 # cover the order for each item
    sum {p in PATTERNS} pieces[i,p] * UsePattern[p] >= demand[i];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ITEMS"] = ["small", "medium", "large"]
ampl.param["width"] = {"small": 3, "medium": 4, "large": 5}
ampl.param["stock_width"] = 10
ampl.param["demand"] = {"small": 5, "medium": 4, "large": 3}

In [5]:
# run.py
from itertools import product

items = ampl.set["ITEMS"].to_list()
width = ampl.param["width"].to_dict()
roll = ampl.param["stock_width"].value()
ranges = [range(int(roll // width[i]) + 1) for i in items]
patterns = [dict(zip(items, c)) for c in product(*ranges)
            if any(c) and sum(width[i] * k for i, k in zip(items, c)) <= roll]
ampl.param["nPatterns"] = len(patterns)
ampl.param["pieces"] = {(i, p): pat[i] for p, pat in enumerate(patterns, 1) for i in items}

ampl.option["relax_integrality"] = 1
ampl.solve(solver="highs", verbose=False)
print(len(patterns), "patterns; LP rolls", ampl.obj["Rolls"].value())
ampl.option["relax_integrality"] = 0
ampl.solve(solver="highs", verbose=False)
use = {p: round(v) for p, v in ampl.var["UsePattern"].to_dict().items() if v > 0.5}
print("integer rolls", ampl.obj["Rolls"].value(),
      {p: (n, patterns[p - 1]) for p, n in use.items()})

11 patterns; LP rolls 4.75
integer rolls 5.0 {4: (2, {'small': 0, 'medium': 1, 'large': 1}), 7: (1, {'small': 1, 'medium': 0, 'large': 1}), 10: (2, {'small': 2, 'medium': 1, 'large': 0})}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. The three nested ranges are written for this
three-item instance:

In [6]:
%%writefile data.dat
set ITEMS := small medium large;
param width := small 3  medium 4  large 5;
param stock_width := 10;
param demand := small 5  medium 4  large 3;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
for {a in 0..floor(stock_width / width['small']),
     b in 0..floor(stock_width / width['medium']),
     c in 0..floor(stock_width / width['large']):
     a + b + c >= 1
     and a * width['small'] + b * width['medium'] + c * width['large'] <= stock_width} {
    let nPatterns := nPatterns + 1;
    let pieces['small', nPatterns] := a;
    let pieces['medium', nPatterns] := b;
    let pieces['large', nPatterns] := c;
}
option relax_integrality 1;
solve;
printf "%d patterns; LP rolls %g\n", nPatterns, Rolls;
option relax_integrality 0;
solve;
display Rolls;
display {p in PATTERNS: UsePattern[p] > 0.5} (UsePattern[p], pieces['small',p],
    pieces['medium',p], pieces['large',p]);

11 patterns; LP rolls 4.75
Rolls = 5

:  UsePattern[p] pieces['small',p] pieces['medium',p] pieces['large',p]    :=
4         2               0                 1                  1
7         1               1                 0                  1
10        2               2                 1                  0
;

